In [ ]:
# --- Imports you’ll need in this notebook ---
import os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

from larch import Group
from larch.io import read_athena
from larch.xafs import feffit, feffit_dataset, TransformGroup

# (Assumes the Cs-fitting functions you pasted are already defined in this notebook:
#  build_params_for_iteration..., get_cs_paths_by_scatterer, enhanced_iteration*_fit, etc.)


In [ ]:
def load_projects_from_athena(prj_files):
    """Load multiple Athena .prj files into a dict of 'project name' -> project.
    Each project has .groups (dict of Group objects)."""
    projects = {}
    for prj in prj_files:
        prj = Path(prj)
        proj = read_athena(str(prj))
        # Name the project by file stem unless you prefer something else
        projects[prj.stem] = proj
    return projects

# Example: point to your Athena projects (.prj)
athena_files = [
    # r"/path/to/your_cs_data_1.prj",
    # r"/path/to/your_cs_data_2.prj",
]

projects = load_projects_from_athena(athena_files)

print(f"Loaded {len(projects)} Athena project(s): {list(projects.keys())}")
# Quick sanity: list first few groups
for pname, proj in projects.items():
    print(f"\nProject: {pname}  |  groups: {len(proj.groups)}")
    for i, gname in enumerate(list(proj.groups.keys())[:5]):
        print("  ", gname)


In [ ]:
# Folder that contains FEFF calculations for Cs absorber (subfolders ok).
# This is what get_cs_paths_by_scatterer() will walk and filter.
feff_base_dir = r"/path/to/feff/Cs"   # <-- change this

assert os.path.isdir(feff_base_dir), f"FEFF base dir not found: {feff_base_dir}"


In [ ]:
# Run your staged fitting workflow.
# Each step augments the path list and stores results on each group.

if len(projects) == 0:
    raise RuntimeError("No projects loaded. Add .prj paths above and re-run.")

print("\n=== Iteration 1: Cs–O ===")
projects = enhanced_iteration1_fit(projects, feff_base_dir=feff_base_dir)

print("\n=== Iteration 2: + Cs–S ===")
projects = enhanced_iteration2_fit(projects, feff_base_dir=feff_base_dir)

print("\n=== Iteration 3: + Cs–I ===")
projects = enhanced_iteration3_fit(projects, feff_base_dir=feff_base_dir)

print("\n=== Iteration 4: + Cs–Pb ===")
projects = enhanced_iteration4_fit(projects, feff_base_dir=feff_base_dir)


In [ ]:
from collections import defaultdict

def summarize_iterations(projects, iterations=(1,2,3,4)):
    rows = []
    for pname, proj in projects.items():
        for gname, g in proj.groups.items():
            for it in iterations:
                fr_attr = f"fit_result_iter{it}"
                path_attr = f"paths_iter{it}"
                if hasattr(g, fr_attr) and hasattr(g, path_attr):
                    fr = getattr(g, fr_attr)
                    paths = getattr(g, path_attr)
                    # Show last-added path label (the winner in that iteration)
                    last_label = getattr(paths[-1], "label", "unknown")
                    rows.append((pname, gname, it, fr.rfactor, last_label))
    return rows

rows = summarize_iterations(projects)
if rows:
    print(f"{'Project':15} {'Group':25} {'Iter':4} {'R-factor':10}  Path")
    print("-"*86)
    for (p, g, it, rfac, label) in sorted(rows, key=lambda r: (r[0], r[1], r[2])):
        print(f"{p:15} {g:25} {it:<4d} {rfac:10.6f}  {label}")
else:
    print("No fit results found. Check earlier cells/log output.")


In [ ]:
def plot_fit_for_group(projects, proj_name, group_name, iteration=4,
                       kmin=3.0, kmax=12.0, kweight=2, rmin=1.0, rmax=4.0):
    """Rebuild the dataset using stored best paths and best-fit params for a group,
    evaluate the model, and plot χ(k) and |χ(R)| vs data."""
    proj = projects[proj_name]
    g = proj.groups[group_name]
    
    # Safety checks
    fr_attr = f"fit_result_iter{iteration}"
    path_attr = f"paths_iter{iteration}"
    if not hasattr(g, fr_attr) or not hasattr(g, path_attr):
        raise ValueError(f"Missing fit result or path list for iteration {iteration} on {proj_name}.{group_name}")
    if not (hasattr(g, "k") and hasattr(g, "chi_tapered")):
        raise ValueError(f"{proj_name}.{group_name} missing 'k' and/or 'chi_tapered' arrays")
    
    fit_result = getattr(g, fr_attr)
    pathlist = list(getattr(g, path_attr))
    
    # Build the transform & data
    trans = TransformGroup(kmin=kmin, kmax=kmax, kweight=kweight, rmin=rmin, rmax=rmax)
    dset = feffit_dataset(data=Group(k=g.k, chi=g.chi_tapered), pathlist=pathlist, transform=trans)
    
    # Recompute model at best-fit parameters without refitting (vary=False is implied by using fixed param values)
    # This runs a quick evaluation to fill dset arrays (model in k and R).
    _ = feffit(fit_result.params, [dset])  # computes model given current params

    # Extract arrays to plot
    kk   = dset.data.k
    chik = dset.data.chi * (kk**kweight)
    chik_model = dset.model * (kk**kweight)  # model in k-domain
    
    rr   = dset.transform.r
    chiR_mag_data = np.abs(dset.transform.chir)
    chiR_mag_model = np.abs(dset.transform.model)

    # --- Plot k-space ---
    plt.figure(figsize=(8, 5))
    plt.plot(kk, chik, label=f"Data (k^{kweight}χ(k))")
    plt.plot(kk, chik_model, label="Model", linestyle="--")
    plt.xlabel("k (Å$^{-1}$)")
    plt.ylabel(f"$k^{kweight}\\chi(k)$")
    plt.title(f"{proj_name}.{group_name} — Iter {iteration} (k-space)")
    plt.legend()
    plt.tight_layout()
    plt.show()

    # --- Plot R-space magnitude ---
    plt.figure(figsize=(8, 5))
    plt.plot(rr, chiR_mag_data, label="|χ(R)| data")
    plt.plot(rr, chiR_mag_model, label="|χ(R)| model", linestyle="--")
    plt.xlabel("R (Å)")
    plt.ylabel("|χ(R)|")
    plt.title(f"{proj_name}.{group_name} — Iter {iteration} (R-space, magnitude)")
    plt.legend()
    plt.tight_layout()
    plt.show()

    # Print a tiny param summary
    print("\nBest-fit parameter snapshot:")
    for name, par in fit_result.params.items():
        try:
            val = par.value
            vary = getattr(par, "vary", False)
            print(f"  {name:12s} = {val: .5f}  (vary={vary})")
        except Exception:
            pass


In [ ]:
# Replace with a real project/group name from your summary above:
example_project = next(iter(projects.keys()))
example_group   = next(iter(projects[example_project].groups.keys()))

plot_fit_for_group(projects, example_project, example_group,
                   iteration=4, kmin=3.0, kmax=12.0, kweight=2, rmin=1.0, rmax=4.0)
